# Практика 3. Airflow: конвейер, который работает сам

**Перед началом:** этот файл только для чтения и общий для всей группы.
Сохраните свою копию: **File → Save Notebook As…** →
`work/students/<ваше_имя>/03_airflow.ipynb`.

Веб-интерфейс Airflow: **http://localhost:8080** — войдите (логин `student`, пароль `student`)
и держите вкладку открытой: сегодня главный экран — там. Ноутбук — место, где мы пишем DAG-и
и проверяем их результаты.

План:
1. Как устроен Airflow
2. Подключение и папка `dags`
3. Первый DAG: три задачи
4. Веб-интерфейс: где что смотреть
5. Порядок задач и параллельность
6. Надёжность: повторы
7. S3 из задач Airflow
8. PostgreSQL из задач Airflow
9. Всё вместе: данные → модели → реестр
10. **Задачи**


## 1. Как устроен Airflow

Airflow — **оркестратор**: он сам ничего не считает и ничего не хранит. Он говорит каждой задаче,
когда начать, следит за порядком и повторяет упавшие шаги. Написать DAG — значит описать конвейер
на обычном Python и положить файл в общую папку `dags`, которую читает планировщик.

Шесть слов, которых хватит на всю практику:

| Слово | Что это |
|---|---|
| **DAG** | весь конвейер: задачи и связи между ними, один `.py`-файл |
| **Task** | один шаг: загрузить, обучить, оценить |
| **Operator** | тип шага: `@task` (Python-функция), `BashOperator` (команда) и другие |
| **Scheduler** | планировщик: следит за расписанием и запускает задачи, у которых готово всё нужное |
| **XCom** | «мелочь между задачами»: числа, строки, ключи — но не датасеты |
| **Connection** | адрес и пароль внешней системы: `s3_local`, `postgres_main` |

Как всё связано на нашем сервере:

```
Jupyter (:8888)                     Airflow (:8080)
вы пишете user1_first.py   →   папка dags → scheduler подхватывает файл (~30 секунд)
                                вы нажимаете ▶ → задачи выполняются по порядку
                                          ↓
      S3 (:9000) файлы · MLflow (:5000) запуски и модели · PostgreSQL (:5432) таблицы
```

**Правила практики** — действуют и на сервере, и локально:

- имя DAG-а, тег и имя файла начинаются с **вашего логина**: `user7_step1`, тег `user7`;
- все DAG-и сегодня с `schedule=None` — запускаем **вручную**, по кнопке (кроме задачи 7);
- в папке `dags` не открываем чужие файлы и ничего не удаляем.


## 2. Подключение и папка `dags`

DAG — это обычный файл в общей папке `dags`: Jupyter пишет в неё, Airflow читает.
Найдём эту папку из ноутбука и заодно подключим S3 и MLflow — они понадобятся в конце,
чтобы проверять результаты задач из Jupyter, как в практикумах 1 и 2.


In [ ]:
from pathlib import Path

# папка dags общая для группы: в ней лежат и DAG-и преподавателя
CANDIDATES = [Path("/home/jovyan/work/dags"), Path("../../dags"), Path("../dags"), Path("dags")]
DAG_DIR = next((p for p in CANDIDATES if p.is_dir() and any(p.glob("*.py"))), None)

assert DAG_DIR is not None, "папка dags не нашлась — спросите путь у преподавателя"
DAGS = sorted(p.name for p in DAG_DIR.glob("*.py"))
print("папка dags:", DAG_DIR.resolve())
print("DAG-файлов в папке:", len(DAGS), "| примеры:", DAGS[:3])


Дальше во всех ячейках `%%writefile` используется путь `/home/jovyan/work/dags`.
Если ваш `DAG_DIR` другой — замените путь в этих ячейках.

И сделайте **один раз** замену по всему ноутбуку: **Edit → Find and Replace**,
`user1` → ваш логин. Логин встречается и в путях `%%writefile`, и в константе `STUDENT`
внутри каждого файла — замена починит всё сразу.


In [ ]:
import os
import boto3
from getpass import getpass

STUDENT = "user1"   # <-- ЗАМЕНИТЕ на свой логин: user1 … user30

S3_URL = os.environ["S3_ENDPOINT_URL"]                     # http://s3:9000 — внутри сервера
S3_PASSWORD = os.environ.get("S3_PASSWORD") or getpass(f"Пароль S3 для {STUDENT}: ")
s3 = boto3.client("s3", endpoint_url=S3_URL,
                  aws_access_key_id=STUDENT, aws_secret_access_key=S3_PASSWORD)

def ls(prefix: str = ""):                                  # тот же помощник, что в практикуме 1
    for obj in s3.list_objects_v2(Bucket=STUDENT, Prefix=prefix).get("Contents", []):
        print(f'{obj["Key"]:<52} {obj["Size"]:>8} байт   {obj["LastModified"]:%Y-%m-%d %H:%M}')

print("мой бакет:", [b["Name"] for b in s3.list_buckets()["Buckets"]])


In [ ]:
import mlflow

os.environ.setdefault("MLFLOW_TRACKING_URI", "http://localhost:5001")
os.environ["MLFLOW_TRACKING_USERNAME"] = STUDENT
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ.get("NB_MLFLOW_PASSWORD") or getpass(f"Пароль MLflow для {STUDENT}: ")

for e in mlflow.search_experiments(max_results=3, order_by=["last_update_time DESC"]):
    print(e.experiment_id, e.name)


## 3. Первый DAG: три задачи

DAG-файл — это обычный Python. Нового всего два декоратора:

- `@dag(...)` — превращает функцию в конвейер: задаёт имя, расписание, тег;
- `@task` — превращает функцию в задачу, а её `return` уходит следующей задаче через **XCom**.

Порядок задаётся **вызовами функций** внизу: `load(transform(extract()))` — значит, сначала
`extract`, потом `transform`, потом `load`. Между задачами передаём только маленькое:
числа, строки, ключи — не датасеты.

Ячейка ниже не выполняет код, а **сохраняет его в файл**: магия `%%writefile` пишет содержимое
ячейки по указанному пути в папку `dags`. Airflow заметит новый файл примерно за 30 секунд.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_first.py
'''Первый DAG: extract → transform → load. Данные — Iris.'''

from datetime import timedelta

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"        # <-- ваш логин (в имени файла он тоже должен быть)


@dag(
    dag_id=f"{STUDENT}_first",                    # имя DAG-а в интерфейсе
    schedule=None,                                # расписание нет — запускаем сами, кнопкой
    start_date=pendulum.datetime(2026, 1, 1),     # с какой даты существует конвейер
    catchup=False,                                # не «догонять» пропущенные даты
    tags=[STUDENT],                               # по тегу удобно фильтровать свои DAG-и
    default_args={"owner": STUDENT, "retries": 1, "retry_delay": timedelta(seconds=15)},
)
def first():
    '''Три задачи по порядку.'''

    @task
    def extract() -> dict:
        '''Шаг 1: берём данные. Тяжёлые импорты — внутри задачи: файл DAG-а перечитывается часто.'''
        from sklearn.datasets import load_iris

        data = load_iris(as_frame=True).frame
        return {                                   # наружу — только маленький словарь
            "rows": len(data),
            "means": data.mean(numeric_only=True).round(2).to_dict(),
        }

    @task
    def transform(info: dict) -> dict:
        '''Шаг 2: из средних выбираем колонку с максимальным средним.'''
        means = info["means"]
        top = max(means, key=means.get)
        return {"rows": info["rows"], "top_feature": top, "top_mean": means[top]}

    @task
    def load(report: dict) -> str:
        '''Шаг 3: печатаем отчёт (в разделе 7 научимся сохранять в S3).'''
        print(f"Iris: {report['rows']} строк; максимальное среднее у «{report['top_feature']}» "
              f"= {report['top_mean']}")
        return "готово"

    load(transform(extract()))                     # вот и весь порядок: extract → transform → load


first()                                            # вызов в конце файла обязателен


### Запускаем в интерфейсе

1. Откройте **http://localhost:8080**.
2. Нажмите **⟳ Refresh**: DAG `user1_first` появится через ~30 секунд после сохранения файла.
   Если вместо DAG-а красная надпись **Import Errors** — в файле ошибка: откройте её,
   там будет номер строки.
3. Переключателем слева от имени включите DAG (снимите с паузы).
4. Справа сверху нажмите **▶ (Trigger DAG run)**.
5. Откройте DAG — вид **Grid**: три квадратика — три задачи; они позеленеют по очереди.
6. Кликните квадратик `load` → **Log** — увидите отчёт, который напечатала задача.


In [ ]:
path = DAG_DIR / f"{STUDENT}_first.py"
print("файл:", path)
print("существует:", path.exists())
print(*[l for l in path.read_text().splitlines() if "dag_id" in l or 'STUDENT =' in l][:2], sep="\n")


**Что произошло.** Scheduler каждые ~30 секунд перечитывает папку `dags` (это называется *парсинг*).
Ваш файл разобрался — DAG появился в списке. По кнопке ▶ создался **DAG run**; scheduler увидел,
что `extract` ни от кого не зависит, и запустил её; `transform` ждала `extract`; `load` — `transform`.
Словари из `return` передались через XCom: откройте квадратик задачи → вкладка **XCom**.

> Изменили код файла? Нажмите ▶ ещё раз — новый запуск возьмёт новый код.
> Кнопка **Clear** у задачи перезапускает её (и всё, что после неё) в том же запуске.


## 4. Веб-интерфейс: где что смотреть

| Статус | Цвет | Что значит |
|---|---|---|
| `success` | зелёный | шаг прошёл |
| `failed` | красный | упал; все следующие ждут |
| `up_for_retry` | оранжевый | упал, но повторы ещё будут |
| `running` / `queued` | зелёный контур | выполняется / в очереди |
| `skipped` | серый | шаг пропущен (так бывает при ветвлении) |

- **Grid** — квадратики по времени: столбец = один DAG run. **Graph** — тот же конвейер стрелками.
- **Log** задачи — главный инструмент отладки: вывод задачи, ошибка со стеком, номер попытки.
- **XCom** (вкладка у задачи) — что задача вернула следующей.
- **Import Errors** (в списке DAG-ов) — если файл не читается: там файл, строка и текст ошибки.
- Свои DAG-и удобно фильтровать по **тегу**: Filter → tags → ваш логин.

> У кого есть доступ к серверу по ssh, то же самое делает командная строка:
> `docker exec -it airflow-scheduler airflow dags list-import-errors` (почему DAG не появился),
> `airflow dags trigger <dag_id>` (запустить), `airflow tasks run <dag> <task> <дата>`.
> Остальным хватает интерфейса.


## 5. Порядок задач и параллельность

Граф в Airflow — направленный и без циклов (DAG = *Directed Acyclic Graph*). Задача стартует,
когда готово всё, от чего она зависит. Независимые задачи идут **параллельно** — как три обучения
на лекции:

```
prepare ──┬──► train(logreg) ──┬──► select_best
          ├──► train(knn) ──────┤
          └──► train(tree) ─────┘
```

В TaskFlow-стиле параллельность получается сама: три вызова `train(...)` в аргументах `select_best(...)`
означают «сначала все трое — потом я».


In [ ]:
%%writefile /home/jovyan/work/dags/user1_parallel.py
'''Параллельные задачи: prepare → три обучения → select_best. Данные — wine.'''

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"


@dag(dag_id=f"{STUDENT}_parallel", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def parallel():

    @task
    def prepare() -> dict:
        '''Готовит данные. Таблицу между задачами не передаём (XCom — только маленькое):
        каждая задача обучения повторит разбиение с тем же random_state и получит те же выборки.'''
        from sklearn.datasets import load_wine
        from sklearn.model_selection import train_test_split

        data = load_wine(as_frame=True).frame
        train, test = train_test_split(data, test_size=0.25, random_state=42, stratify=data["target"])
        return {"train_rows": len(train), "test_rows": len(test)}

    @task
    def train(model_name: str, sizes: dict) -> dict:
        '''Обучает одну модель. Три такие задачи идут параллельно.'''
        from sklearn.datasets import load_wine
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import train_test_split
        from sklearn.neighbors import KNeighborsClassifier
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.tree import DecisionTreeClassifier

        data = load_wine(as_frame=True).frame
        train, test = train_test_split(data, test_size=0.25, random_state=42, stratify=data["target"])

        models = {
            "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
            "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
            "tree": DecisionTreeClassifier(max_depth=3, random_state=42),
        }
        model = models[model_name].fit(train.drop(columns="target"), train["target"])
        accuracy = model.score(test.drop(columns="target"), test["target"])
        return {"model": model_name, "accuracy": round(accuracy, 4)}

    @task
    def select_best(r1: dict, r2: dict, r3: dict) -> dict:
        results = [r1, r2, r3]
        best = max(results, key=lambda r: r["accuracy"])
        print("результаты:", [(r["model"], r["accuracy"]) for r in results])
        print("лучший:", best["model"])
        return best

    sizes = prepare()
    select_best(train("logreg", sizes), train("knn", sizes), train("tree", sizes))


parallel()


Запустите DAG и посмотрите **Graph**: три `train` стартуют одновременно, `select_best` ждёт все три.
В Grid видно, как три квадратика зеленеют вместе, а четвёртый загорается после них.


## 6. Надёжность: повторы

Ночной сбой — норма: не ответила база, моргнула сеть. Хорошая задача сама **повторяется**:
`retries=4` — после падения попробовать ещё 4 раза, `retry_delay` — пауза между попытками.

Повторы лечат только **временные** сбои. Опечатка в коде упадёт все пять раз — и это правильно:
после одинаковых ошибок человек должен увидеть красный квадратик, а не двадцать пустых писем.

Сделаем задачу, которая честно падает в половине случаев:


In [ ]:
%%writefile /home/jovyan/work/dags/user1_retry.py
'''Задача-врун: падает в половине случаев. Смотрим, как Airflow повторяет попытки.'''

import random
from datetime import timedelta

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"


@dag(dag_id=f"{STUDENT}_retry", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def retry():

    @task(retries=4, retry_delay=timedelta(seconds=10))   # упала — повторить ещё 4 раза
    def flaky() -> str:
        if random.random() < 0.5:
            raise RuntimeError("временный сбой: «база не ответила» (учебная ошибка)")
        return "ok"

    @task
    def report(status: str):
        print("итог:", status)

    report(flaky())


retry()


Запустите DAG несколько раз (▶) и смотрите **Grid**:

- оранжевый `up_for_retry` — попытка упала, через 10 секунд будет новая;
- в **Log** каждой попытки видно её номер и причину падения;
- если все 5 попыток упали (вероятность 0,5⁵ ≈ 3%), задача станет красной, `report` не стартует,
  и весь DAG run станет красным.

**Сравните с cron.** Обычный планировщик после падения шага 2 всё равно выполнил бы шаги 3–5
на старых данных — и никто бы не узнал. Airflow остановил конвейер и показал, где именно упало.


## 7. S3 из задач Airflow

Паролей в коде DAG-а нет: адрес и ключи S3 уже лежат в **подключении** `s3_local`
(в локальном стеке оно задано переменной окружения; на сервере — Admin → Connections). В задаче достаточно `S3Hook(aws_conn_id="s3_local")`.

| boto3 (практикум 1) | S3Hook (Airflow) |
|---|---|
| `put_object(Body=строка)` | `load_string(строка, key, bucket_name)` |
| `put_object(Body=байты)` | `load_bytes(байты, key, bucket_name, replace=True)` |
| `get_object(...)["Body"].read()` | `read_key(key, bucket_name)` / `download_file(key, bucket_name)` — вернёт путь |
| `list_objects_v2(Prefix=...)` | `list_keys(prefix, bucket_name)` |
| `head_object` | `check_for_key(key, bucket_name)` |

Два правила из лекции:

1. **В XCom — ключ, не данные.** Датасет живёт в S3, следующей задаче передаётся строка-ключ.
2. **Дата запуска — часть ключа.** Параметр `ds` в функции задачи — дата запуска (`2026-11-06`):
   каждый запуск кладёт файлы в свою «папку» и не перезаписывает чужие.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_s3hook.py
'''DAG и S3: prepare → check. Ключ — по дате запуска ds.'''

import io

import pendulum
from airflow.decorators import dag, task
from airflow.providers.amazon.aws.hooks.s3 import S3Hook

STUDENT = "user1"
BUCKET = STUDENT                                  # бакет называется вашим логином


@dag(dag_id=f"{STUDENT}_s3hook", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def s3hook_demo():

    @task
    def prepare(ds=None) -> str:
        '''ds — дата запуска (например 2026-11-06): у каждого запуска своя «папка» в S3.'''
        from sklearn.datasets import load_iris

        df = load_iris(as_frame=True).frame
        buf = io.BytesIO()
        df.to_parquet(buf, index=False)            # DataFrame -> parquet (буфер в памяти)

        key = f"airflow/{ds}/iris.parquet"
        S3Hook(aws_conn_id="s3_local").load_bytes(  # байты -> S3, ключи берёт из Connection
            buf.getvalue(), key=key, bucket_name=BUCKET, replace=True)
        return key                                 # дальше передаём только ключ

    @task
    def check(key: str):
        hook = S3Hook(aws_conn_id="s3_local")
        print("файл на месте:", hook.check_for_key(key, bucket_name=BUCKET))
        print("в airflow/ лежит:", hook.list_keys(prefix="airflow/", bucket_name=BUCKET))

    check(prepare())


s3hook_demo()


In [ ]:
# запустите после того, как DAG user1_s3hook отработал (квадратики зелёные)
ls("airflow/")


Каждый новый запуск пишет в «папку» своей даты. Если запусков в один день несколько,
`replace=True` перезапишет файл; чтобы этого избежать, в ключ добавляют время запуска —
сделаем это в задаче 7.


## 8. PostgreSQL из задач Airflow

Табличные результаты (метрики по запускам) удобно держать в базе — их можно спросить SQL-ом.
Подключение `postgres_main` уже настроено. У каждого — своя **схема**, называется логином,
как бакет в S3: `user1.runs`.

| Метод | Что делает |
|---|---|
| `hook.run(sql)` | выполнить SQL (`CREATE TABLE …`) |
| `hook.insert_rows(таблица, rows)` | вставить строки списком кортежей |
| `hook.get_records(sql)` | прочитать строки |


In [ ]:
%%writefile /home/jovyan/work/dags/user1_pg.py
'''DAG и PostgreSQL: обучаем модель, метрику — в таблицу, читаем обратно.'''

import pendulum
from airflow.decorators import dag, task
from airflow.providers.postgres.hooks.postgres import PostgresHook

STUDENT = "user1"


def pg():
    return PostgresHook(postgres_conn_id="postgres_main")


@dag(dag_id=f"{STUDENT}_pg", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def pg_demo():

    @task
    def init() -> str:
        '''Своя схема и таблица метрик (IF NOT EXISTS — можно запускать много раз).'''
        pg().run(f'CREATE SCHEMA IF NOT EXISTS "{STUDENT}"')
        pg().run(f'''CREATE TABLE IF NOT EXISTS "{STUDENT}".runs (
                          ds TEXT, task TEXT, metric DOUBLE PRECISION,
                          created_at TIMESTAMP DEFAULT now())''')
        return "ok"

    @task
    def train(ok: str, ds=None) -> float:
        from sklearn.datasets import load_iris
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import train_test_split
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler

        data = load_iris(as_frame=True).frame
        train_df, test_df = train_test_split(data, test_size=0.25, random_state=42,
                                             stratify=data["target"])
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
        model.fit(train_df.drop(columns="target"), train_df["target"])
        accuracy = model.score(test_df.drop(columns="target"), test_df["target"])

        pg().insert_rows(f'"{STUDENT}".runs', rows=[(ds, "train_logreg", round(accuracy, 4))])
        return round(accuracy, 4)

    @task
    def read_back(accuracy: float):
        rows = pg().get_records(
            f'SELECT ds, task, metric FROM "{STUDENT}".runs ORDER BY created_at DESC LIMIT 5')
        print("accuracy этой задачи:", accuracy)
        print("последние строки в базе:")
        for row in rows:
            print("  ", row)

    read_back(train(init()))


pg_demo()


Запустите DAG и откройте лог задачи `read_back`: в таблице — строка с датой запуска и метрикой.
Каждый запуск **дописывает** строку, так что в базе растёт история метрик — сравните с практикумом 1,
где история жила только в списках файлов.


## 9. Всё вместе: данные → модели → реестр

Соберём конвейер с лекции: `prepare → две модели параллельно → лучшая → реестр MLflow → прогноз`.
Инструменты делят работу: **Airflow** запускает и следит за порядком, **S3** хранит данные,
**MLflow** записывает запуски и хранит версии модели с меткой `champion`.

Один новый шаг — пароль MLflow. В Jupyter мы вводили его через `getpass`; задача Airflow
спросить пароль не может. В локальном стеке Variable `user1_mlflow_password` **уже создана** — там заглушка, потому что локальный MLflow без пароля. На сервере положите в неё настоящий пароль: **Admin → Variables → «+».

В коде достаём `Variable.get(...)`: пароль не попадает ни в файл DAG-а, ни в Git.

> Не забудьте про Find and Replace: `user1` → ваш логин — тогда и ключ Variable поменяется сам.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_full.py
'''Полный конвейер: данные → S3 → две модели параллельно (обе в MLflow) → лучшая в реестр → прогноз.'''

import os

import pendulum
from airflow.decorators import dag, task
from airflow.models import Variable

STUDENT = "user1"
EXPERIMENT = f"{STUDENT}_airflow"           # эксперимент в MLflow
MODEL_NAME = f"{STUDENT}_airflow_model"     # модель в реестре


def mlflow_on():
    '''MLflow из задачи: адрес — окружение, логин — ваш, пароль — из Airflow Variable.'''
    import mlflow

    os.environ.setdefault("MLFLOW_TRACKING_URI", "http://mlflow:5000")
    os.environ["MLFLOW_TRACKING_USERNAME"] = STUDENT
    os.environ["MLFLOW_TRACKING_PASSWORD"] = Variable.get(f"{STUDENT}_mlflow_password",
                                                          default_var="")
    return mlflow


def log_sklearn_model(mlflow, model, X_example):
    '''Сохраняет sklearn-модель в текущий запуск (формат skops — как в практикуме 2).'''
    import skops.io as sio
    from mlflow.models import infer_signature

    return mlflow.sklearn.log_model(
        model, name="model",
        signature=infer_signature(X_example, model.predict(X_example)),
        input_example=X_example.head(3),
        skops_trusted_types=sio.get_untrusted_types(data=sio.dumps(model)),
    )


@dag(dag_id=f"{STUDENT}_full", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def full():

    @task
    def prepare(ds=None) -> str:
        '''Breast Cancer → S3 (parquet). Наружу — ключ.'''
        import io
        from airflow.providers.amazon.aws.hooks.s3 import S3Hook
        from sklearn.datasets import load_breast_cancer

        df = load_breast_cancer(as_frame=True).frame
        buf = io.BytesIO()
        df.to_parquet(buf, index=False)
        key = f"airflow/{ds}/cancer.parquet"
        S3Hook(aws_conn_id="s3_local").load_bytes(buf.getvalue(), key=key,
                                                  bucket_name=STUDENT, replace=True)
        return key

    @task
    def train(model_name: str, key: str) -> dict:
        '''Обучает одну модель и пишет запуск в MLflow. Две задачи train идут параллельно.'''
        import pandas as pd
        from airflow.providers.amazon.aws.hooks.s3 import S3Hook
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import roc_auc_score
        from sklearn.model_selection import train_test_split
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler

        local = S3Hook(aws_conn_id="s3_local").download_file(key=key, bucket_name=STUDENT)   # вернёт путь к файлу
        df = pd.read_parquet(local)

        train_df, test_df = train_test_split(df, test_size=0.25, random_state=42,
                                             stratify=df["target"])
        models = {
            "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
            "random_forest": RandomForestClassifier(random_state=42),
        }
        X_test, y_test = test_df.drop(columns="target"), test_df["target"]

        model = models[model_name].fit(train_df.drop(columns="target"), train_df["target"])
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

        mlflow = mlflow_on()
        mlflow.set_experiment(EXPERIMENT)
        with mlflow.start_run(run_name=model_name) as run:
            mlflow.log_param("model", model_name)
            mlflow.log_metric("roc_auc", round(auc, 4))
            log_sklearn_model(mlflow, model, X_test)
        return {"model": model_name, "roc_auc": round(auc, 4), "run_id": run.info.run_id}

    @task
    def select_best(r1: dict, r2: dict) -> dict:
        best = max([r1, r2], key=lambda r: r["roc_auc"])
        print(f"{r1['model']}: {r1['roc_auc']} | {r2['model']}: {r2['roc_auc']}"
              f" → лучший: {best['model']}")
        return best

    @task
    def register_and_predict(best: dict):
        from sklearn.datasets import load_breast_cancer

        mlflow = mlflow_on()
        version = mlflow.register_model(f"runs:/{best['run_id']}/model", MODEL_NAME)
        mlflow.MlflowClient().set_registered_model_alias(MODEL_NAME, "champion", version.version)

        champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
        X, _ = load_breast_cancer(return_X_y=True, as_frame=True)
        print(f"реестр: {MODEL_NAME}, новая версия {version.version} ({best['model']}) — метка @champion")
        print("прогноз чемпиона для 3 строк:", champion.predict(X.head(3)).tolist())

    key = prepare()
    register_and_predict(select_best(train("logreg", key), train("random_forest", key)))


full()


In [ ]:
# запустите после того, как DAG user1_full отработал (все квадратики зелёные)
try:
    runs = mlflow.search_runs(experiment_names=[f"{STUDENT}_airflow"],
                              order_by=["metrics.roc_auc DESC"], max_results=4)
    runs[["tags.mlflow.runName", "params.model", "metrics.roc_auc"]].head()
except Exception as e:
    print("запусков ещё нет — сначала запустите DAG", f"{STUDENT}_full:", e)


**Один запуск — кто что делает:**

| Задача | Airflow | S3 | MLflow |
|---|---|---|---|
| `prepare` | запустил | положил `cancer.parquet` | — |
| `train` ×2 | запустил **параллельно** | — | два запуска: параметры, `roc_auc`, модели |
| `select_best` | дождался обеих | — | — |
| `register_and_predict` | запустил | — | новая версия + метка `champion` |

После каждой задачи Airflow кладёт её лог в бакет `airflow-logs`, а статусы хранит в PostgreSQL.

Запустите DAG ещё раз: в реестре появится следующая версия, и метка переедет на неё —
а код при этом не меняется: он загружает модель по адресу `models:/…@champion`.


## Шпаргалка

| Что | Код |
|---|---|
| конвейер | `@dag(dag_id=..., schedule=None, catchup=False, tags=[STUDENT])` + вызов функции в конце |
| задача | `@task` над функцией; порядок — вызовами: `c(b(a()))` |
| повторы | `@task(retries=3, retry_delay=timedelta(seconds=30))` |
| дата запуска | параметр `ds=None` в функции задачи |
| S3 | `S3Hook(aws_conn_id="s3_local")`: `load_string` · `load_bytes` · `read_key` · `download_file` · `list_keys` · `check_for_key` |
| PostgreSQL | `PostgresHook(postgres_conn_id="postgres_main")`: `run` · `insert_rows` · `get_records` |
| секрет | `Variable.get("имя")` — значение задаётся в Admin → Variables |
| классический стиль | `PythonOperator(task_id=..., python_callable=...)`; порядок: `a >> b`, `[a, b] >> c` |
| расписание | `schedule=None` — вручную · `"@daily"` — каждый день · cron `"0 4 * * *"` — ежедневно в 4:00 **UTC** |

---

# 10. Задачи

Каждая задача — **новый файл** в `dags`: `user1_step1.py`, `user1_step2.py`…
Логин — ваш: в имени файла, в `dag_id`, в теге и в константе `STUDENT`.
Запуск — кнопкой ▶; результат смотрите в Grid/Log, а из Jupyter — через `ls()` и MLflow.

Скелеты уже рабочие: `...` (многоточие) — валидный Python, поэтому DAG появится в интерфейсе
и позеленеет, даже пока логики нет. Заполняйте задачи по очереди и запускайте после каждого шага.


### Задача 1. Первый DAG
Сделайте `user1_step1.py`: `extract → transform → load` на датасете **wine**.
`extract` возвращает число строк и средние значения колонок; `transform` выбирает колонку
с максимальным средним; `load` печатает отчёт. Запустите и покажите лог `load` и XCom задачи
`transform`.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_step1.py
'''Задача 1: extract → transform → load на датасете wine.'''

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"


@dag(dag_id=f"{STUDENT}_step1", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def step1():

    @task
    def extract() -> dict:
        # ваш код: load_wine(as_frame=True).frame -> {"rows": ..., "means": {...}}
        ...

    @task
    def transform(info: dict) -> dict:
        # ваш код: выбрать колонку с максимальным средним
        ...

    @task
    def load(report: dict) -> str:
        # ваш код: напечатать отчёт
        ...

    # ваш код: соединить задачи вызовами


step1()


### Задача 2. Параллельность и повторы
`user1_step2.py` на **iris**: `prepare` (разбиение train/test, `random_state=42`) → три
**параллельных** обучения (`logreg`, `knn`, `tree`) → `select_best` по accuracy на тесте.
Одной из задач обучения поставьте `retries=3` и добавьте в неё «временный сбой»
(`random.random() < 0.3 → raise`). В Grid найдите оранжевый квадратик и повтор в логе.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_step2.py
'''Задача 2: параллельные обучения и повторы. Данные — iris.'''

import random
from datetime import timedelta

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"


@dag(dag_id=f"{STUDENT}_step2", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def step2():

    @task
    def prepare() -> dict:
        # ваш код: iris, train_test_split(..., random_state=42, stratify=...) -> размеры
        ...

    @task
    def train(model_name: str, sizes: dict) -> dict:
        # ваш код: словарь моделей logreg/knn/tree; в одной ветке — «временный сбой»
        # не забудьте @task(retries=3, retry_delay=timedelta(seconds=10)) над этой задачей
        ...

    @task
    def select_best(r1: dict, r2: dict, r3: dict) -> dict:
        # ваш код: лучшая по accuracy
        ...

    # ваш код: подготовить, запустить три train параллельно, выбрать лучшего


step2()


### Задача 3. Данные в S3 и PostgreSQL
`user1_step3.py`: `prepare` кладёт iris в S3 как parquet по ключу `airflow/{ds}/iris.parquet`
и возвращает ключ; `train` читает его (`download_file`), обучает логрег и пишет accuracy
в таблицу `"{STUDENT}".runs` (схему и таблицу создайте в этой же задаче, как в разделе 8);
`read_back` печатает последние 5 строк. Проверьте `ls("airflow/")` из Jupyter.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_step3.py
'''Задача 3: данные в S3, метрика в PostgreSQL.'''

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"
BUCKET = STUDENT


@dag(dag_id=f"{STUDENT}_step3", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def step3():

    @task
    def prepare(ds=None) -> str:
        # ваш код: iris -> parquet -> S3 по ключу airflow/{ds}/iris.parquet; вернуть ключ
        ...

    @task
    def train(key: str, ds=None) -> float:
        # ваш код: скачать parquet, обучить логрег, CREATE SCHEMA/TABLE IF NOT EXISTS,
        # insert_rows с (ds, "step3_train", accuracy); вернуть accuracy
        ...

    @task
    def read_back(accuracy: float):
        # ваш код: последние 5 строк из "{STUDENT}".runs
        ...

    # ваш код: порядок prepare -> train -> read_back


step3()


### Задача 4. Модель и артефакты в S3
`user1_step4.py` на **breast cancer**: `prepare` → train/test parquet в S3; `train` обучает
логрег и кладёт в S3 `model.joblib`, `metrics.json` (accuracy и roc_auc) и `confusion_matrix.png`;
`predict` загружает модель из S3 и печатает прогноз для 5 строк теста. MLflow здесь не нужен —
только S3. Проверьте `ls()` и откройте картинку из Jupyter (как в практикуме 1).


In [ ]:
%%writefile /home/jovyan/work/dags/user1_step4.py
'''Задача 4: модель, метрики и график — всё в S3. Данные — breast cancer.'''

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"
BUCKET = STUDENT


@dag(dag_id=f"{STUDENT}_step4", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def step4():

    @task
    def prepare(ds=None) -> dict:
        # ваш код: train/test -> parquet -> S3 (airflow/{ds}/step4/...); вернуть {"train": ключ, "test": ключ}
        ...

    @task
    def train(keys: dict, ds=None) -> dict:
        # ваш код: обучить логрег; model.joblib (joblib.dump в буфер), metrics.json,
        # confusion_matrix.png (matplotlib, backend Agg) -> S3; вернуть ключи + метрики
        ...

    @task
    def predict(artifacts: dict):
        # ваш код: скачать model.joblib и тест, predict для 5 строк, напечатать
        ...

    # ваш код: порядок


step4()


### Задача 5. Четыре модели и реестр
`user1_step5.py` — конвейер с лекции (`teacher_step5_mlflow`): `prepare` (breast cancer → S3) →
**четыре параллельных** обучения (logreg, random_forest, knn, tree), каждое пишет запуск в MLflow
(параметры, `roc_auc`, модель) → `select_best` → регистрация лучшей как `f"{STUDENT}_step5_model"`
с меткой `champion` и прогноз для 5 строк. Проверьте в интерфейсе MLflow: эксперимент,
четыре запуска, вкладка **Models**.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_step5.py
'''Задача 5: четыре модели параллельно, каждая — запуск в MLflow, лучшая — в реестр.'''

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"
BUCKET = STUDENT
EXPERIMENT = f"{STUDENT}_step5"
MODEL_NAME = f"{STUDENT}_step5_model"


# подсказка: функции mlflow_on() и log_sklearn_model(mlflow, model, X_example) — из раздела 9


@dag(dag_id=f"{STUDENT}_step5", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def step5():

    @task
    def prepare(ds=None) -> str:
        # ваш код: breast cancer -> parquet -> S3; вернуть ключ
        ...

    @task
    def train_and_log(model_name: str, key: str) -> dict:
        # ваш код: 4 модели в словаре, roc_auc, запуск в MLflow c log_model;
        # вернуть {"model", "roc_auc", "run_id"}
        ...

    @task
    def select_best(r1: dict, r2: dict, r3: dict, r4: dict) -> dict:
        # ваш код: лучшая по roc_auc (+ печатать всех по убыванию)
        ...

    @task
    def register_and_predict(best: dict):
        # ваш код: register_model + метка champion + прогноз для 5 строк
        ...

    # ваш код: prepare -> 4 параллельных train_and_log -> select_best -> register_and_predict


step5()


### Задача 6 ★. Ворота качества
`user1_step6.py`: как задача 5, но перед регистрацией узнайте `roc_auc` текущего чемпиона
(метка → версия → запуск → метрика) и регистрируйте новую версию с переносом метки **только если
претендент строго лучше**. Запустите DAG дважды: во второй раз метка должна остаться на месте —
и в логе должно быть написано, почему. Так модели выходят в продакшн без человека.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_step6.py
'''Задача 6 ★: ворота качества — метка champion переезжает, только если претендент лучше.'''

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"
EXPERIMENT = f"{STUDENT}_step6"
MODEL_NAME = f"{STUDENT}_step6_model"


@dag(dag_id=f"{STUDENT}_step6", schedule=None,
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def step6():

    # ваш код: prepare и train_and_log — как в задаче 5 (можно обучить 2 модели)

    @task
    def quality_gate(best: dict):
        # ваш код: клиент MLflow; champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
        # roc_auc чемпиона: client.get_run(champ.run_id).data.metrics["roc_auc"]
        # если чемпиона ещё нет — выпустить первую версию; иначе сравнить и решить
        ...

    # ваш код: порядок


step6()


### Задача 7. Расписание
Сделайте `user1_step7.py` — лёгкий DAG с расписанием `*/10 * * * *` (каждые 10 минут).
Его единственная задача пишет файл-«пульс» `airflow/heartbeat/{ts}.txt` (из `ts` уберите двоеточия).
**Включите** DAG переключателем и **не нажимайте** ▶: запуски появляются сами. Посмотрите на время
запусков: Airflow живёт в UTC — почему они на 3 часа «раньше» московских? Закончив наблюдение,
поставьте DAG на паузу.


In [ ]:
%%writefile /home/jovyan/work/dags/user1_step7.py
'''Задача 7: DAG с расписанием — файл-«пульс» каждые 10 минут.'''

import pendulum
from airflow.decorators import dag, task

STUDENT = "user1"
BUCKET = STUDENT


@dag(dag_id=f"{STUDENT}_step7",
     schedule=None,   # ваш код: */10 * * * *
     start_date=pendulum.datetime(2026, 1, 1), catchup=False, tags=[STUDENT])
def step7():

    @task
    def heartbeat(ts=None) -> str:
        # ваш код: ts — момент запуска (2026-11-06T10:20:00+00:00);
        # ключ airflow/heartbeat/<ts без двоеточий>.txt, содержимое — "пульс {ts}"
        ...

    heartbeat()


step7()


### Задача 8. Интерфейс
1. В `user1_step2` найдите повтор: сколько попыток в итоге сделала задача со «сбоем»?
2. Откройте XCom задачи `select_best` — что в нём лежит и почему не датасет?
3. Сломайте нарочно один свой файл (уберите скобку) → Refresh → найдите **Import Errors**,
   прочитайте сообщение, почините файл.
4. Отфильтруйте список DAG-ов по своему тегу: сколько ваших DAG-ов накопилось?


---

**Готово?** Вы прошли весь путь с первого слайда лекции: данные → обучение → оценка → реестр →
прогноз — и всё это запускается кнопкой, а по расписанию будет запускаться само.
Дальше — проект: свой DAG на своих данных, с параллельными шагами, метриками в MLflow
и лучшей моделью в реестре.
